##### 1 — Validar o ambiente e importar bibliotecas

In [ ]:
import os
import sys
from pathlib import Path

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        "Selecione /usr/bin/python3.11 como interpretador do notebook."
    )

if not Path("/projects/python-packages/setup-completo.json").exists():
    raise RuntimeError(
        "A preparação das bibliotecas não foi concluída. "
        "Consulte /projects/workspace-bootstrap.log."
    )

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import SparkClient
from kubeflow.spark.types.types import Driver
from pyspark.sql import functions as F

import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("Ambiente pronto.")

##### 2 — Configurar o Spark exclusivo e os dados

In [ ]:
NAMESPACE = os.environ.get(
    "SPARK_NAMESPACE",
    "redhat-ods-applications",
)

IMAGEM_SPARK = "quay.io/opendatahub/data-processing:Spark-v4.0.1"
SERVICE_ACCOUNT_DRIVER = "spark-operator-spark"

NUM_EXECUTORES = 2
CPU_POR_EXECUTOR = "1"
MEMORIA_POR_EXECUTOR = "2Gi"

CPU_DRIVER = "1"
MEMORIA_DRIVER = "1Gi"

SPARK_CONF = {
    "spark.sql.shuffle.partitions": "4",
    "spark.sql.adaptive.enabled": "true",
    "spark.dynamicAllocation.enabled": "false",
}

NUM_TRANSACOES = 100_000
NUM_CLIENTES = 1_000
LIMITE_SUSPEITA = 9_500

assert NUM_EXECUTORES > 0
assert NUM_TRANSACOES > 0
assert NUM_CLIENTES > 0

print("Namespace:", NAMESPACE)
print("Executores:", NUM_EXECUTORES)
print("CPU por executor:", CPU_POR_EXECUTOR)
print("Memória por executor:", MEMORIA_POR_EXECUTOR)

##### 3 — Criar e conectar ao Spark Connect exclusivo

In [ ]:
if globals().get("spark") is not None:
    print("Reutilizando a sessão criada por este notebook.")

elif globals().get("criacao_iniciada", False):
    raise RuntimeError(
        "Já houve uma tentativa de criação neste kernel. "
        "Confira os SparkConnects no OpenShift antes de tentar novamente, "
        "pois a tentativa anterior pode ter criado um recurso."
    )

else:
    criacao_iniciada = True

    client = SparkClient(
        backend_config=KubernetesBackendConfig(
            namespace=NAMESPACE,
        )
    )

    print(
        f"Criando Spark Connect com {NUM_EXECUTORES} executores...",
        flush=True,
    )

    spark = client.connect(
        num_executors=NUM_EXECUTORES,
        resources_per_executor={
            "cpu": CPU_POR_EXECUTOR,
            "memory": MEMORIA_POR_EXECUTOR,
        },
        driver=Driver(
            image=IMAGEM_SPARK,
            service_account=SERVICE_ACCOUNT_DRIVER,
            resources={
                "cpu": CPU_DRIVER,
                "memory": MEMORIA_DRIVER,
            },
        ),
        spark_conf=SPARK_CONF,
        timeout=300,
        connect_timeout=120,
    )

    print("Sessão conectada.")

print("Spark remoto:", spark.version)
print("Registros de teste:", spark.range(1_000).count())

##### 4 — Gerar transações fictícias

In [ ]:
transacoes = (
    spark.range(NUM_TRANSACOES)
    .withColumnRenamed("id", "transacao_id")
    .withColumn(
        "cliente_id",
        (F.col("transacao_id") % NUM_CLIENTES).cast("int"),
    )
    .withColumn(
        "valor",
        F.round(F.rand(seed=42) * 10_000 + 10, 2),
    )
    .withColumn(
        "canal",
        F.when(
            F.col("transacao_id") % 3 == 0,
            "mobile",
        ).otherwise("internet"),
    )
    .withColumn(
        "suspeita",
        (F.col("valor") > LIMITE_SUSPEITA).cast("int"),
    )
)

transacoes.show(10, truncate=False)

total = transacoes.count()
assert total == NUM_TRANSACOES

print(f"Total de transações: {total:,}")

##### 5 — Resumir por canal

In [ ]:
resumo = (
    transacoes.groupBy("canal")
    .agg(
        F.count("*").alias("total_transacoes"),
        F.round(F.sum("valor"), 2).alias("valor_total"),
        F.round(F.avg("valor"), 2).alias("ticket_medio"),
        F.sum("suspeita").alias("total_suspeitas"),
    )
    .withColumn(
        "percentual_suspeitas",
        F.round(
            F.col("total_suspeitas")
            / F.col("total_transacoes") * 100,
            2,
        ),
    )
)

resumo.orderBy("canal").show(truncate=False)

##### 6 — Gerar features por cliente

In [ ]:
features = transacoes.groupBy("cliente_id").agg(
    F.count("*").alias("quantidade_transacoes"),
    F.round(F.sum("valor"), 2).alias("valor_total"),
    F.round(F.avg("valor"), 2).alias("valor_medio"),
    F.max("valor").alias("maior_transacao"),
    F.sum("suspeita").alias("quantidade_suspeitas"),
)

features.orderBy(
    F.desc("valor_total"),
    F.asc("cliente_id"),
).show(10, truncate=False)

##### 7 — Consultar usando SQL

In [ ]:
transacoes.createOrReplaceTempView("transacoes_ficticias")

spark.sql("""
    SELECT
        canal,
        COUNT(*) AS quantidade_suspeitas,
        ROUND(AVG(valor), 2) AS valor_medio
    FROM transacoes_ficticias
    WHERE suspeita = 1
    GROUP BY canal
    ORDER BY canal
""").show(truncate=False)

##### 8 — Exibir gráficos

In [ ]:
resumo_local = resumo.orderBy("canal").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

resumo_local.plot.bar(
    x="canal",
    y="total_transacoes",
    ax=axes[0],
    legend=False,
    rot=0,
)

axes[0].set_title("Transações por canal")
axes[0].set_xlabel("Canal")
axes[0].set_ylabel("Quantidade")

resumo_local.plot.bar(
    x="canal",
    y="percentual_suspeitas",
    ax=axes[1],
    legend=False,
    rot=0,
    color="darkorange",
)

axes[1].set_title("Transações suspeitas")
axes[1].set_xlabel("Canal")
axes[1].set_ylabel("Percentual (%)")

plt.tight_layout()
plt.show()

##### 9 — Encerrar a sessão do notebook

In [ ]:
if globals().get("spark") is not None:
    spark.stop()
    spark = None

print("Sessão encerrada.")
print("O recurso Spark Connect e seus pods continuam no OpenShift.")